# Mingyu's SafeTune setup
GitHub / Hugging Face: **xtnayk**. Prepared November 2, 2026.

This notebook is a setup check, not the hackathon experiment. It runs the official
small-model quickstart, then tests Tiny Aya with a benign prompt through SafeTune.
The official quickstart demonstrates refusal **ablation**, not safety repair.
Qwen is only an installation check; the team brief specifies Cohere models for the project.

**Before running:** upload this notebook to [Colab](https://colab.research.google.com/),
then select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
Use a Python 3.12+ runtime. Free GPU availability varies.

Run cells in order. For Tiny Aya, sign in as `xtnayk` and accept the access conditions at
[CohereLabs/tiny-aya-global](https://huggingface.co/CohereLabs/tiny-aya-global).
Create a read token in [Hugging Face settings](https://huggingface.co/settings/tokens),
then put it in Colab's **Secrets** panel as `HF_TOKEN` and enable notebook access.
Do not paste the token into a code cell or chat.

No model upload, paid API, fine-tuning, or account creation is performed here.
The Qwen quickstart passed on a Tesla T4 on November 2, 2026. Tiny Aya was not run. The current local project uses Aya Expanse 8B; this notebook remains the Colab fallback, not validation that unquantized 8B fits a T4.


## 1. Check the Colab runtime


In [ ]:
import sys, os, json, subprocess
from pathlib import Path
try:
    import google.colab
except ImportError:
    raise RuntimeError("Run this notebook in Google Colab; the installation cell changes only that runtime.")
if sys.version_info < (3, 12):
    raise RuntimeError(f"SafeTune requires Python 3.12+. This runtime is {sys.version.split()[0]}.")
ROOT = Path('/content/lexsi-safetune')
SOURCE = ROOT / 'vendor' / 'SafeTune'
RESULTS = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
COMMIT = '26f59e6d1bdcacafe5f2be3b8d982fb214369659'
os.environ['WANDB_MODE'] = 'disabled'
print('Python:', sys.version.split()[0])
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)


## 2. Install the reviewed SafeTune source
The exact source revision is pinned. Its metadata currently declares version 0.1.8;
the source commit is the reproducibility reference. SafeTune's README recommends
removing Colab's `torchao` before importing SafeTune. No optional vLLM, Unsloth,
or CircuitKIT stack is installed. Installation can take several minutes.

All ML checks run in separate Python processes so changed packages are not mixed
with previously imported versions, and GPU memory is released between models.


In [ ]:
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/Lexsi-Labs/SafeTune.git', str(SOURCE)], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
if actual != COMMIT:
    raise RuntimeError('Existing checkout differs. Use a fresh Colab runtime rather than overwriting it.')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE)], check=True)
print('Installed source:', actual)


## 3. Verify imports and capture installed versions


In [ ]:
check = subprocess.run([sys.executable, '-c', "import json, torch, safetune, importlib.metadata as md\nfrom safetune.runner.utils import load_model, load_tok\nfrom safetune.core.eval.pipeline import GenerationConfig, TransformersBackend\nfrom safetune._refusal_helpers import _get_decoder_layers\nfrom safetune.steer import extract_refusal_direction, RefusalDirectionModel\nnames = ['safetune', 'torch', 'transformers', 'accelerate', 'peft', 'trl', 'huggingface-hub']\nprint(json.dumps({'packages': {name: md.version(name) for name in names}, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}, indent=2))\nif not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before proceeding.')\n"], text=True, capture_output=True)
(RESULTS / 'imports.log').write_text(check.stdout + check.stderr)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError('Import/GPU verification failed. Keep imports.log for debugging.')
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RESULTS / 'packages.txt').write_text(freeze)


## 4. Prepare the isolated runner
Each run resolves and records an immutable model revision, saves a JSON report,
and keeps detailed logs. Model weights are downloaded to Colab only when a run starts.
The Tiny Aya run uses batch size 1 and a short generation; available memory is still
checked by the actual run, not assumed. No safety score is inferred from a nonempty answer.


In [ ]:
RUNNER = ROOT / 'smoke_check.py'
RUNNER.write_text('#!/usr/bin/env python3\n"""Run one isolated setup check; save evidence, never publish or train."""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nSAFETUNE_COMMIT = "26f59e6d1bdcacafe5f2be3b8d982fb214369659"\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", required=True)\n    parser.add_argument("--source", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--device", choices=("auto", "cpu", "cuda", "mps"), default="auto")\n    parser.add_argument("--official-quickstart", action="store_true")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    report = {\n        "started_utc": datetime.now(timezone.utc).isoformat(),\n        "model": args.model,\n        "mode": "official_quickstart" if args.official_quickstart else "benign_generation",\n        "status": "running",\n        "stage": "environment",\n        "python": sys.version,\n        "scope": "Setup check only; not evidence of safety improvement.",\n    }\n    # A terminated process must never leave an earlier successful report behind.\n    (args.output / "report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    try:\n        if sys.version_info < (3, 12):\n            raise RuntimeError("SafeTune requires Python 3.12 or newer.")\n        report["safetune_commit"] = subprocess.check_output(\n            ["git", "-C", str(args.source), "rev-parse", "HEAD"], text=True\n        ).strip()\n        if report["safetune_commit"] != SAFETUNE_COMMIT:\n            raise RuntimeError("SafeTune source does not match the reviewed revision.")\n        dirty = subprocess.check_output(\n            ["git", "-C", str(args.source), "status", "--porcelain", "--untracked-files=no"],\n            text=True,\n        )\n        report["safetune_tracked_files_clean"] = not bool(dirty.strip())\n        if dirty.strip():\n            raise RuntimeError("SafeTune has edited tracked files; use a clean checkout for this setup check.")\n        report["packages"] = {\n            name: importlib.metadata.version(name)\n            for name in ("safetune", "torch", "transformers", "accelerate", "peft", "trl", "huggingface-hub")\n        }\n        import torch\n        import safetune\n        from safetune.config import resolve_device\n        from huggingface_hub import HfApi, snapshot_download\n\n        if not Path(safetune.__file__).resolve().is_relative_to(args.source.resolve()):\n            raise RuntimeError("Imported SafeTune is not the pinned source checkout. Re-run installation.")\n        device = resolve_device(args.device)\n        report["device"] = device\n        if device == "mps" and not torch.backends.mps.is_available():\n            raise RuntimeError("PyTorch cannot access this Mac\'s Metal GPU.")\n        if device == "cuda" and not torch.cuda.is_available():\n            raise RuntimeError("PyTorch cannot access a CUDA GPU.")\n        if device == "cuda":\n            report["gpu"] = torch.cuda.get_device_name(0)\n            report["vram_gib"] = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)\n        if args.model.lower().startswith("coherelabs/") and device == "cpu":\n            raise RuntimeError("Use --device mps on Apple Silicon or --device cuda for this Aya check.")\n        if device == "mps":\n            report["mps_recommended_max_gib"] = round(torch.mps.recommended_max_memory() / 2**30, 2)\n\n        report["stage"] = "model_download"\n        info = HfApi().model_info(args.model)\n        report["model_revision"] = info.sha\n        if not info.sha:\n            raise RuntimeError("Could not resolve an immutable model revision.")\n        snapshot = snapshot_download(\n            args.model,\n            revision=info.sha,\n            allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja"],\n        )\n        if args.official_quickstart:\n            report["stage"] = "official_quickstart"\n            script = args.source / "examples/quickstart/quickstart.py"\n            with (args.output / "quickstart.log").open("w") as log:\n                result = subprocess.run(\n                    [sys.executable, "-u", str(script), "--model", snapshot,\n                     "--device", device, "--max-new-tokens", "32"],\n                    stdout=log, stderr=subprocess.STDOUT,\n                )\n            report["quickstart_exit_code"] = result.returncode\n            if result.returncode:\n                raise RuntimeError("Official quickstart failed; inspect quickstart.log.")\n        else:\n            report["stage"] = "model_load"\n            from safetune.runner.utils import load_model, load_tok\n            from safetune._refusal_helpers import _get_decoder_layers\n            from safetune.core.eval.pipeline import GenerationConfig, TransformersBackend\n\n            safetune.configure(device=device, seed=42)\n            tokenizer = load_tok(snapshot)\n            load_started = time.perf_counter()\n            model = load_model(snapshot)\n            if device == "mps":\n                torch.mps.synchronize()\n            elif device == "cuda":\n                torch.cuda.synchronize()\n            report["model_load_seconds"] = round(time.perf_counter() - load_started, 3)\n            model.eval()\n            layers = _get_decoder_layers(model)\n            if not len(layers):\n                raise RuntimeError("SafeTune did not discover any decoder layers.")\n            report["decoder_layers"] = len(layers)\n            report["model_type"] = model.config.model_type\n            report["dtype"] = str(next(model.parameters()).dtype)\n            report["model_device"] = str(next(model.parameters()).device)\n            prompt = "Explain in one sentence why keeping experiment logs is useful."\n            token_ids = tokenizer.apply_chat_template(\n                [{"role": "user", "content": prompt}],\n                tokenize=True, add_generation_prompt=True, return_dict=False,\n            )\n            report["templated_prompt_tokens"] = len(token_ids)\n            report["stage"] = "benign_generation"\n            backend = TransformersBackend(\n                model, tokenizer,\n                config=GenerationConfig(max_new_tokens=32, batch_size=1, temperature=0, seed=42),\n            )\n            generation_started = time.perf_counter()\n            answer = backend.generate([prompt])[0]\n            if device == "mps":\n                torch.mps.synchronize()\n            elif device == "cuda":\n                torch.cuda.synchronize()\n            report["generation_seconds"] = round(time.perf_counter() - generation_started, 3)\n            if not answer.strip():\n                raise RuntimeError("Model generated an empty answer.")\n            report["benign_answer"] = answer\n            if device == "cuda":\n                report["peak_allocated_vram_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)\n            elif device == "mps":\n                report["mps_allocated_gib_after_generation"] = round(torch.mps.current_allocated_memory() / 2**30, 2)\n                report["mps_driver_allocated_gib_after_generation"] = round(torch.mps.driver_allocated_memory() / 2**30, 2)\n        report["status"] = "passed"\n        report["stage"] = "complete"\n    except Exception as exc:\n        report["status"] = "failed"\n        report["error_type"] = type(exc).__name__\n        report["error"] = str(exc)\n        (args.output / "error.log").write_text(traceback.format_exc())\n    finally:\n        report["finished_utc"] = datetime.now(timezone.utc).isoformat()\n        target = args.output / "report.json"\n        target.write_text(json.dumps(report, indent=2) + "\\n")\n        print(f"{report[\'status\'].upper()}: {args.model}; stage={report[\'stage\']}")\n        print(f"Evidence: {target}")\n    return 0 if report["status"] == "passed" else 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n')
print('Runner ready:', RUNNER)


In [ ]:
def run_check(name, model, official=False):
    from datetime import datetime, timezone
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = RESULTS / f'{name}-{attempt}'
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(RUNNER), '--model', model,
               '--source', str(SOURCE), '--output', str(output)]
    if official:
        command.append('--official-quickstart')
    print(f'Running {name}. Downloads and the first run can take several minutes.')
    with (output / 'runner.log').open('w') as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    report_path = output / 'report.json'
    report = json.loads(report_path.read_text()) if report_path.exists() else {'status': 'failed', 'stage': 'process'}
    if result.returncode or report.get('status') != 'passed':
        report['status'] = 'failed' if result.returncode else 'incomplete'
        print(json.dumps(report, indent=2))
        raise RuntimeError(f'{name} failed or was interrupted. Inspect logs in {output}; download them in the last cell.')
    print(json.dumps(report, indent=2))
    return report


## 5. Run the official small-model quickstart
This uses `Qwen/Qwen2.5-0.5B-Instruct` and SafeTune's unmodified quickstart script.
It extracts and ablates a refusal direction. Outputs and its small English
refusal-phrase counts are in `quickstart.log`; they are **not multilingual evaluation**.
No Hugging Face token is required for this public model.


In [ ]:
qwen_report = run_check('01_qwen_quickstart', 'Qwen/Qwen2.5-0.5B-Instruct', official=True)


## 6. Verify your Tiny Aya access before downloading weights
Finish the access and Colab Secrets steps at the top first. This verifies that
`HF_TOKEN` belongs to `xtnayk` and can download the gated model's small config file.
It never prints or writes the token. A 401/403 response means to check the account,
accepted access conditions, and the token's gated-repository read permission.


In [ ]:
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access, then rerun this cell.') from None
subprocess.run([sys.executable, '-c', "from huggingface_hub import HfApi, hf_hub_download\napi = HfApi()\nusername = api.whoami()['name']\nif username != 'xtnayk': raise RuntimeError(f'Token belongs to {username}, expected xtnayk.')\ninfo = api.model_info('CohereLabs/tiny-aya-global')\nhf_hub_download('CohereLabs/tiny-aya-global', 'config.json', revision=info.sha)\nprint('Tiny Aya config access verified for', username)\n"], check=True)


## 7. Run Tiny Aya's SafeTune compatibility check
Loads one Tiny Aya model through SafeTune, locates decoder layers, formats the chat
prompt, and generates a short benign answer. Records dtype, model revision,
prompt length, GPU memory, package versions, and the run's actual status.
This does not fine-tune or repair the model, or establish multilingual safety.


In [ ]:
tiny_report = run_check('02_tiny_aya_smoke', 'CohereLabs/tiny-aya-global')


## 8. Optional: try the same official quickstart on Tiny Aya
Run only after the compatibility check passes. Set the flag below to `True`
when you are ready for a longer run. The upstream quickstart performs a layer sweep
and may need more GPU and host memory than the smoke check. Its handwritten
generation code also has a possible duplicate-BOS formatting issue for Tiny Aya;
keep this result as an upstream quickstart trial, not a scientific baseline.
Capture any failure rather than treating it as proof the model is unsupported.


In [ ]:
RUN_TINY_OFFICIAL_QUICKSTART = False
if RUN_TINY_OFFICIAL_QUICKSTART:
    tiny_quickstart_report = run_check('03_tiny_aya_official_quickstart', 'CohereLabs/tiny-aya-global', official=True)
else:
    print('Optional Tiny Aya quickstart skipped.')


## 9. Download evidence before disconnecting
Run this even after a failure. Colab storage is temporary. The archive contains
run reports and logs, not model weights or the `HF_TOKEN` secret. Review logs
before publishing them. You can bring the archive to your next setup/debugging session.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(ROOT / 'lexsi-setup-results'), 'zip', RESULTS)
files.download(archive)


## After setup
- Bring actual pass/fail results and any errors to Aish. Do not report unrun steps as complete.
- Confirm SafeTune scope and which languages you can validate before building experiments.
- Read CircuitKIT's [getting started](https://circuitkit.lexsi.ai/) and its
  refusal-localization tutorial. Tiny Aya/Cohere compatibility still needs a separate check.
- For later Tiny Aya training, upstream warns about its long chat preamble:
  leave `max_len` automatic or use at least 512, then verify supervised tokens.
- Full safety evaluation needs separate harmful/benign sets, utility metrics, and
  language-aware judging. The quickstart's English phrase matcher is insufficient.

Sources: [SafeTune at the pinned revision](https://github.com/Lexsi-Labs/SafeTune/tree/26f59e6d1bdcacafe5f2be3b8d982fb214369659),
[Tiny Aya model card](https://huggingface.co/CohereLabs/tiny-aya-global),
[CircuitKIT documentation](https://circuitkit.lexsi.ai/).
